# Módulo 6 — Delta Sharing: El Protocolo de Integración SAP

**Arquitectura de la clase:**
```
[TRIAL workspace]                    [FREE EDITION workspace]
 Proveedor (FP&A / área de negocio)   Receptor (Summa / Área de datos)
 ─────────────────────────────────    ──────────────────────────────────
 budget_ventas        ──────────────→  gold_sales_kpis   (ventas reales SAP)
 forecast_demanda     ──────────────→  gold_customer_360 (clientes SAP)
 tasa_cambio_usd_cop  ──────────────→
                                       → Análisis Real vs Presupuesto
                                       → Varianza por organización
                                       → Revenue normalizado a COP
```

**Conexión con BDC:**
> SAP Business Data Cloud hace exactamente esto — publica data products
> via Delta Sharing y Databricks los consume como receptor.
> La bidireccionalidad: resultados ML regresan a BDC como nuevos data products.

## PASO 1 — Crear las tablas de negocio en el Trial

Simulamos el área de FP&A de Grupo Argos publicando:
- **budget_ventas**: presupuesto de ventas por mes/organización/moneda
- **forecast_demanda**: proyección de órdenes (modelo ML XGBoost)
- **tasa_cambio**: USD/EUR a COP para normalizar revenue

In [0]:
# Configuración del catálogo Trial
TRIAL_CATALOG = "workspace"
TRIAL_SCHEMA  = "default"

spark.sql(f"USE CATALOG {TRIAL_CATALOG}")
spark.sql(f"USE SCHEMA {TRIAL_SCHEMA}")
print(f"✅ Usando: {TRIAL_CATALOG}.{TRIAL_SCHEMA}")

In [0]:
# ── Datos de presupuesto de ventas ──
from pyspark.sql.types import *
from pyspark.sql import Row

budget_data = [
    Row(year=2020,month=1,sales_org="1000",currency="COP",budget_revenue=3745200.5,budget_orders=6,budget_avg_order=624200.08,scenario="BASE"),
    Row(year=2020,month=1,sales_org="1000",currency="USD",budget_revenue=1876543.2,budget_orders=5,budget_avg_order=375308.64,scenario="BASE"),
    Row(year=2020,month=1,sales_org="1000",currency="EUR",budget_revenue=1423100.0,budget_orders=4,budget_avg_order=355775.0,scenario="CONSERVADOR"),
    Row(year=2020,month=1,sales_org="2000",currency="COP",budget_revenue=2234500.75,budget_orders=5,budget_avg_order=446900.15,scenario="BASE"),
    Row(year=2020,month=1,sales_org="2000",currency="USD",budget_revenue=2987600.3,budget_orders=6,budget_avg_order=497933.38,scenario="OPTIMISTA"),
    Row(year=2020,month=1,sales_org="2000",currency="EUR",budget_revenue=2456700.0,budget_orders=5,budget_avg_order=491340.0,scenario="BASE"),
    Row(year=2020,month=2,sales_org="1000",currency="COP",budget_revenue=3876543.0,budget_orders=7,budget_avg_order=553806.14,scenario="BASE"),
    Row(year=2020,month=2,sales_org="1000",currency="USD",budget_revenue=1945000.0,budget_orders=5,budget_avg_order=389000.0,scenario="BASE"),
    Row(year=2020,month=2,sales_org="1000",currency="EUR",budget_revenue=1534200.5,budget_orders=4,budget_avg_order=383550.13,scenario="BASE"),
    Row(year=2020,month=2,sales_org="2000",currency="COP",budget_revenue=2356700.0,budget_orders=6,budget_avg_order=392783.33,scenario="CONSERVADOR"),
    Row(year=2020,month=2,sales_org="2000",currency="USD",budget_revenue=3123400.8,budget_orders=7,budget_avg_order=446200.11,scenario="BASE"),
    Row(year=2020,month=2,sales_org="2000",currency="EUR",budget_revenue=2567800.2,budget_orders=6,budget_avg_order=427966.7,scenario="BASE"),
    Row(year=2020,month=3,sales_org="1000",currency="COP",budget_revenue=4123456.0,budget_orders=7,budget_avg_order=589065.14,scenario="BASE"),
    Row(year=2020,month=3,sales_org="1000",currency="USD",budget_revenue=2034500.0,budget_orders=6,budget_avg_order=339083.33,scenario="BASE"),
    Row(year=2020,month=3,sales_org="1000",currency="EUR",budget_revenue=1645300.0,budget_orders=5,budget_avg_order=329060.0,scenario="OPTIMISTA"),
    Row(year=2020,month=3,sales_org="2000",currency="COP",budget_revenue=2478900.5,budget_orders=6,budget_avg_order=413150.08,scenario="BASE"),
    Row(year=2020,month=3,sales_org="2000",currency="USD",budget_revenue=3267800.3,budget_orders=7,budget_avg_order=466828.61,scenario="BASE"),
    Row(year=2020,month=3,sales_org="2000",currency="EUR",budget_revenue=2689700.0,budget_orders=6,budget_avg_order=448283.33,scenario="BASE"),
    Row(year=2021,month=6,sales_org="1000",currency="COP",budget_revenue=5234567.0,budget_orders=9,budget_avg_order=581618.56,scenario="BASE"),
    Row(year=2021,month=6,sales_org="1000",currency="USD",budget_revenue=2567800.5,budget_orders=7,budget_avg_order=366828.64,scenario="BASE"),
    Row(year=2021,month=6,sales_org="1000",currency="EUR",budget_revenue=1934500.0,budget_orders=5,budget_avg_order=386900.0,scenario="BASE"),
    Row(year=2021,month=6,sales_org="2000",currency="COP",budget_revenue=3123456.7,budget_orders=7,budget_avg_order=446208.1,scenario="OPTIMISTA"),
    Row(year=2021,month=6,sales_org="2000",currency="USD",budget_revenue=3987600.4,budget_orders=8,budget_avg_order=498450.05,scenario="BASE"),
    Row(year=2021,month=6,sales_org="2000",currency="EUR",budget_revenue=3234500.0,budget_orders=7,budget_avg_order=462071.43,scenario="CONSERVADOR"),
    Row(year=2022,month=9,sales_org="1000",currency="COP",budget_revenue=5876543.2,budget_orders=10,budget_avg_order=587654.32,scenario="BASE"),
    Row(year=2022,month=9,sales_org="1000",currency="USD",budget_revenue=2934500.0,budget_orders=7,budget_avg_order=419214.29,scenario="BASE"),
    Row(year=2022,month=9,sales_org="1000",currency="EUR",budget_revenue=2123400.5,budget_orders=6,budget_avg_order=353900.08,scenario="BASE"),
    Row(year=2022,month=9,sales_org="2000",currency="COP",budget_revenue=3456700.8,budget_orders=8,budget_avg_order=432087.6,scenario="BASE"),
    Row(year=2022,month=9,sales_org="2000",currency="USD",budget_revenue=4234500.3,budget_orders=9,budget_avg_order=470500.03,scenario="OPTIMISTA"),
    Row(year=2022,month=9,sales_org="2000",currency="EUR",budget_revenue=3567800.0,budget_orders=8,budget_avg_order=445975.0,scenario="BASE"),
    Row(year=2023,month=3,sales_org="1000",currency="COP",budget_revenue=6234567.5,budget_orders=10,budget_avg_order=623456.75,scenario="BASE"),
    Row(year=2023,month=3,sales_org="1000",currency="USD",budget_revenue=3123400.0,budget_orders=8,budget_avg_order=390425.0,scenario="BASE"),
    Row(year=2023,month=3,sales_org="1000",currency="EUR",budget_revenue=2345600.3,budget_orders=6,budget_avg_order=390933.38,scenario="BASE"),
    Row(year=2023,month=3,sales_org="2000",currency="COP",budget_revenue=3789600.7,budget_orders=9,budget_avg_order=421066.74,scenario="CONSERVADOR"),
    Row(year=2023,month=3,sales_org="2000",currency="USD",budget_revenue=4567800.2,budget_orders=10,budget_avg_order=456780.02,scenario="BASE"),
    Row(year=2023,month=3,sales_org="2000",currency="EUR",budget_revenue=3890500.0,budget_orders=8,budget_avg_order=486312.5,scenario="BASE"),
    Row(year=2024,month=1,sales_org="1000",currency="COP",budget_revenue=6543210.0,budget_orders=11,budget_avg_order=594837.27,scenario="BASE"),
    Row(year=2024,month=1,sales_org="1000",currency="USD",budget_revenue=3234500.5,budget_orders=8,budget_avg_order=404312.56,scenario="OPTIMISTA"),
    Row(year=2024,month=1,sales_org="1000",currency="EUR",budget_revenue=2456700.0,budget_orders=6,budget_avg_order=409450.0,scenario="BASE"),
    Row(year=2024,month=1,sales_org="2000",currency="COP",budget_revenue=3987600.8,budget_orders=9,budget_avg_order=442955.64,scenario="BASE"),
    Row(year=2024,month=1,sales_org="2000",currency="USD",budget_revenue=4789600.3,budget_orders=10,budget_avg_order=478960.03,scenario="BASE"),
    Row(year=2024,month=1,sales_org="2000",currency="EUR",budget_revenue=4023400.0,budget_orders=9,budget_avg_order=447044.44,scenario="CONSERVADOR"),
    Row(year=2024,month=6,sales_org="1000",currency="COP",budget_revenue=7234567.0,budget_orders=12,budget_avg_order=602880.58,scenario="BASE"),
    Row(year=2024,month=6,sales_org="1000",currency="USD",budget_revenue=3567800.2,budget_orders=9,budget_avg_order=396422.24,scenario="BASE"),
    Row(year=2024,month=6,sales_org="1000",currency="EUR",budget_revenue=2678900.5,budget_orders=7,budget_avg_order=382700.07,scenario="OPTIMISTA"),
    Row(year=2024,month=6,sales_org="2000",currency="COP",budget_revenue=4234500.7,budget_orders=10,budget_avg_order=423450.07,scenario="BASE"),
    Row(year=2024,month=6,sales_org="2000",currency="USD",budget_revenue=5023400.0,budget_orders=11,budget_avg_order=456672.73,scenario="BASE"),
    Row(year=2024,month=6,sales_org="2000",currency="EUR",budget_revenue=4345600.3,budget_orders=10,budget_avg_order=434560.03,scenario="BASE"),
    Row(year=2024,month=12,sales_org="1000",currency="COP",budget_revenue=7890600.0,budget_orders=13,budget_avg_order=606969.23,scenario="OPTIMISTA"),
    Row(year=2024,month=12,sales_org="1000",currency="USD",budget_revenue=3890500.5,budget_orders=9,budget_avg_order=432277.83,scenario="BASE"),
    Row(year=2024,month=12,sales_org="1000",currency="EUR",budget_revenue=2987600.0,budget_orders=7,budget_avg_order=426800.0,scenario="BASE"),
    Row(year=2024,month=12,sales_org="2000",currency="COP",budget_revenue=4567800.8,budget_orders=11,budget_avg_order=415254.62,scenario="BASE"),
    Row(year=2024,month=12,sales_org="2000",currency="USD",budget_revenue=5345600.3,budget_orders=12,budget_avg_order=445466.69,scenario="CONSERVADOR"),
    Row(year=2024,month=12,sales_org="2000",currency="EUR",budget_revenue=4678900.0,budget_orders=10,budget_avg_order=467890.0,scenario="BASE"),
]

budget_schema = StructType([
    StructField("year",             IntegerType(), False),
    StructField("month",            IntegerType(), False),
    StructField("sales_org",        StringType(),  False),
    StructField("currency",         StringType(),  False),
    StructField("budget_revenue",   DoubleType(),  True),
    StructField("budget_orders",    IntegerType(), True),
    StructField("budget_avg_order", DoubleType(),  True),
    StructField("scenario",         StringType(),  True),
])

df_budget = spark.createDataFrame(budget_data, schema=budget_schema)
(df_budget.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{TRIAL_CATALOG}.{TRIAL_SCHEMA}.budget_ventas"))

print(f"✅ budget_ventas creado: {df_budget.count()} registros")
df_budget.show(5)

In [0]:
# ── Tasa de cambio USD/EUR → COP ──
fx_data = [
    Row(year=2020,month=1, tasa_usd_cop=3457.23, tasa_eur_cop=3823.45, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2020,month=6, tasa_usd_cop=3746.89, tasa_eur_cop=4123.67, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2020,month=12,tasa_usd_cop=3432.56, tasa_eur_cop=4198.34, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2021,month=1, tasa_usd_cop=3523.78, tasa_eur_cop=4267.89, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2021,month=6, tasa_usd_cop=3745.23, tasa_eur_cop=4456.78, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2021,month=12,tasa_usd_cop=3987.45, tasa_eur_cop=4523.12, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2022,month=1, tasa_usd_cop=3934.56, tasa_eur_cop=4478.23, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2022,month=6, tasa_usd_cop=4234.78, tasa_eur_cop=4567.89, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2022,month=12,tasa_usd_cop=4756.34, tasa_eur_cop=5123.45, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2023,month=1, tasa_usd_cop=4823.67, tasa_eur_cop=5234.56, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2023,month=6, tasa_usd_cop=4234.89, tasa_eur_cop=4678.34, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2023,month=12,tasa_usd_cop=3934.23, tasa_eur_cop=4345.67, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2024,month=1, tasa_usd_cop=3978.45, tasa_eur_cop=4389.23, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2024,month=6, tasa_usd_cop=4123.67, tasa_eur_cop=4456.78, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2024,month=11,tasa_usd_cop=4356.89, tasa_eur_cop=4678.45, fuente="Banco de la República", tipo="Promedio mensual"),
    Row(year=2024,month=12,tasa_usd_cop=4489.34, tasa_eur_cop=4823.56, fuente="Banco de la República", tipo="Promedio mensual"),
]

fx_schema = StructType([
    StructField("year",         IntegerType(), False),
    StructField("month",        IntegerType(), False),
    StructField("tasa_usd_cop", DoubleType(),  True),
    StructField("tasa_eur_cop", DoubleType(),  True),
    StructField("fuente",       StringType(),  True),
    StructField("tipo",         StringType(),  True),
])

df_fx = spark.createDataFrame(fx_data, schema=fx_schema)
(df_fx.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{TRIAL_CATALOG}.{TRIAL_SCHEMA}.tasa_cambio_usd_cop"))

print(f"✅ tasa_cambio_usd_cop creado: {df_fx.count()} registros")
df_fx.show(5)

## PASO 2 — Crear el Share como Proveedor

El trial workspace actúa como **FP&A de Grupo Argos** publicando
datos de presupuesto para que el equipo de datos (Free Edition) los consuma.

In [0]:
%sql
-- Otorgar permisos sobre el catálogo y schema
GRANT USE CATALOG ON CATALOG workspace TO `info@ecotickets.co`;
GRANT USE SCHEMA ON SCHEMA workspace.default TO `info@ecotickets.co`;
GRANT SELECT ON TABLE workspace.default.budget_ventas TO `info@ecotickets.co`;
GRANT SELECT ON TABLE workspace.default.tasa_cambio_usd_cop TO `info@ecotickets.co`;

In [0]:
%sql
-- Crear el share de datos financieros
CREATE SHARE IF NOT EXISTS fpya_grupo_argos
  COMMENT 'Presupuesto y tasas de cambio — FP&A Grupo Argos para análisis Real vs Budget';

-- Agregar tabla de presupuesto
ALTER SHARE fpya_grupo_argos
  ADD TABLE workspace.default.budget_ventas
  AS budget.ventas_presupuesto;

-- Agregar tasa de cambio
ALTER SHARE fpya_grupo_argos
  ADD TABLE workspace.default.tasa_cambio_usd_cop
  AS referencia.tasa_cambio;

-- Verificar
SHOW SHARES;

In [0]:
%sql
-- Ver el detalle del share creado
DESCRIBE SHARE fpya_grupo_argos;

In [0]:
%sql
-- Ver tablas dentro del share
SHOW ALL IN SHARE fpya_grupo_argos;

## PASO 3 — Crear el Recipient (receptor autorizado)

En producción con BDC: SAP registra el workspace de Databricks del cliente
como recipient. Aquí simulamos que el Free Edition de Summa es el receptor.

In [0]:
%sql
-- Crear el recipient — genera un activation link
CREATE RECIPIENT IF NOT EXISTS summa_datos_sap
  COMMENT 'Workspace Free Edition de Summa — equipo de Data Engineering';

-- Ver el activation link (se usa para conectar desde el receptor)
DESCRIBE RECIPIENT summa_datos_sap;

In [0]:
%sql
-- Otorgar acceso al recipient sobre el share
GRANT SELECT ON SHARE fpya_grupo_argos TO RECIPIENT summa_datos_sap;

-- Verificar permisos
SHOW GRANTS ON SHARE fpya_grupo_argos;

In [0]:
%sql
-- Ver providers disponibles
SHOW PROVIDERS;

-- Ver shares del proveedor Free Edition
SHOW SHARES IN PROVIDER summa;

-- Crear catálogo local apuntando al share
CREATE CATALOG IF NOT EXISTS sap_gold_shared
  USING SHARE summa.sap_analytics_share;

-- Consultar las tablas Gold SAP — zero copy!
SHOW TABLES IN CATALOG sap_gold_shared;

## PASO 4 — Consumir el Share como Receptor

Ahora cambiamos de rol: somos el **equipo de datos de Summa** (Free Edition)
recibiendo los datos de FP&A via Delta Sharing.

En el Free Edition ejecutar:
```sql
-- Registrar el proveedor con el activation link
CREATE PROVIDER fpya_argos_provider
  USING DELTA SHARING
  RECIPIENT_PROFILE_STR '<pegar activation link aquí>';

-- Ver shares disponibles del proveedor
SHOW SHARES IN PROVIDER fpya_argos_provider;

-- Crear catálogo local apuntando al share
CREATE CATALOG IF NOT EXISTS fpya_shared
  USING SHARE fpya_argos_provider.fpya_grupo_argos;

-- Consultar — zero copy!
SELECT * FROM fpya_shared.budget.ventas_presupuesto LIMIT 10;
```

## PASO 5 — Análisis Real vs Presupuesto

Este análisis se ejecuta en el **Free Edition** cruzando:
- `gold_sales_kpis` — ventas reales de SAP
- `fpya_shared.budget.ventas_presupuesto` — presupuesto via Delta Sharing

In [0]:
# ── ANÁLISIS REAL VS BUDGET — ejecutar desde el TRIAL ──
spark.sql("""
SELECT
    r.year,
    r.month,
    r.sales_org,
    r.currency,
    -- Ventas reales (SAP via OpenSharing desde Free Edition)
    r.num_orders                                              AS ordenes_reales,
    ROUND(r.total_revenue, 2)                                 AS ventas_reales,
    -- Presupuesto (tabla local del trial — creada en este mismo notebook)
    b.budget_orders                                           AS ordenes_budget,
    ROUND(b.budget_revenue, 2)                                AS ventas_budget,
    b.scenario                                                AS escenario,
    -- Análisis de varianza
    ROUND(r.total_revenue - b.budget_revenue, 2)              AS varianza_cop,
    ROUND((r.total_revenue / b.budget_revenue - 1) * 100, 1)  AS varianza_pct,
    -- Semáforo de cumplimiento
    CASE
        WHEN r.total_revenue >= b.budget_revenue              THEN '✅ Cumple'
        WHEN r.total_revenue >= b.budget_revenue * 0.90       THEN '⚠️  Cerca (>90%)'
        WHEN r.total_revenue >= b.budget_revenue * 0.75       THEN '🟡 Gap moderado'
        ELSE                                                       '❌ Gap crítico'
    END                                                       AS estado_cumplimiento
FROM sap_gold_shared.sales.monthly_kpis    r   -- ventas reales SAP via OpenSharing
JOIN workspace.default.budget_ventas       b   -- presupuesto local del trial
  ON r.year      = b.year
 AND r.month     = b.month
 AND r.sales_org = b.sales_org
 AND r.currency  = b.currency
ORDER BY r.year DESC, r.month DESC, varianza_pct ASC
""").show(20, truncate=False)

In [0]:
%sql
-- ── Normalización de revenue a COP usando tasa de cambio ──
-- Todos los valores en una sola moneda para comparar correctamente
-- Ejecutar desde el TRIAL

SELECT
    r.year,
    r.month,
    r.sales_org,
    r.currency,
    ROUND(r.total_revenue, 2)                                 AS ventas_moneda_original,
    -- Conversión a COP
    ROUND(
        CASE r.currency
            WHEN 'COP' THEN r.total_revenue
            WHEN 'USD' THEN r.total_revenue * fx.tasa_usd_cop
            WHEN 'EUR' THEN r.total_revenue * fx.tasa_eur_cop
        END, 2
    )                                                         AS ventas_cop,
    fx.tasa_usd_cop,
    fx.tasa_eur_cop
FROM sap_gold_shared.sales.monthly_kpis         r   -- ventas reales SAP via OpenSharing
LEFT JOIN workspace.default.tasa_cambio_usd_cop fx  -- tasa de cambio local del trial
       ON r.year  = fx.year
      AND r.month = fx.month
ORDER BY r.year DESC, r.month DESC, ventas_cop DESC;

## PASO 6 — Conexión con BDC

Lo que hicieron en esta clase es **exactamente** lo que hace SAP BDC:

| Esta demo | BDC en producción |
|-----------|------------------|
| Trial workspace = proveedor | SAP BDC = proveedor |
| `CREATE SHARE fpya_grupo_argos` | BDC crea el share automáticamente al activar un data product |
| `CREATE RECIPIENT summa_datos_sap` | BDC Connect registra el workspace Databricks del cliente |
| `budget_ventas` vía Delta Sharing | `I_SalesOrder` data product vía Delta Sharing |
| Free Edition = receptor | Tu Databricks de producción = receptor |
| Análisis Real vs Budget | ML, forecasting, scoring sobre datos SAP en vivo |

**La diferencia:**
> En BDC el proveedor es SAP — ellos gestionan los data products.
> Tú solo configuras BDC Connect una vez (one-time setup)
> y consumes los data products como tablas Delta Sharing en Unity Catalog.

## Resumen de comandos clave

```sql
-- Como PROVEEDOR (Trial / BDC)
CREATE SHARE nombre_share;
ALTER SHARE nombre_share ADD TABLE catalogo.schema.tabla AS alias.nombre;
CREATE RECIPIENT nombre_receptor;
GRANT SELECT ON SHARE nombre_share TO RECIPIENT nombre_receptor;
SHOW SHARES;
DESCRIBE SHARE nombre_share;
DESCRIBE RECIPIENT nombre_receptor;   -- aquí está el activation link

-- Como RECEPTOR (Free Edition / Tu Databricks)
CREATE PROVIDER nombre_proveedor
  USING DELTA SHARING
  RECIPIENT_PROFILE_STR '<activation_link>';
SHOW SHARES IN PROVIDER nombre_proveedor;
CREATE CATALOG nombre_catalogo
  USING SHARE nombre_proveedor.nombre_share;
SELECT * FROM nombre_catalogo.alias.nombre;
```